<a href="https://colab.research.google.com/github/EstherMan05/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-25%20%E2%80%94%20Cleaning%20Gauntlet%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
before = len(df)
df = df.drop_duplicates().copy()
removed = before - len(df)

log('duplicates', 'dropped exact duplicate rows', removed)
print('rows after dedup:', len(df))

[duplicates] dropped exact duplicate rows (15 row(s))
rows after dedup: 300


### TODO 2 — clean `price` -> float

In [4]:
df['price'] = (
    df['price'].astype(str)
    .str.replace('$', '', regex=False)
    .str.strip()
    .astype(float)
)

assert df['price'].dtype == float
log('price', 'stripped $ signs and whitespace, converted to float', len(df))

[price] stripped $ signs and whitespace, converted to float (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')
missing = df['qty'].isna().sum()
negative = (df['qty'] < 0).sum()

df_before_drop = df.copy()   # snapshot for the revenue before/after comparison

before_drop = len(df)
df = df[df['qty'] >= 1].copy()
dropped = before_drop - len(df)

log('qty missing', f'dropped {missing} row(s) with missing quantity', missing)
log('qty negative', f'dropped {negative} row(s) with negative quantity (refunds)', negative)

# revenue before vs after this step, for the write-up
before_revenue = (df_before_drop['qty'] * df_before_drop['price']).sum()
after_revenue = (df['qty'] * df['price']).sum()
print('revenue before drop:', round(before_revenue, 2))
print('revenue after drop:', round(after_revenue, 2))

[qty missing] dropped 12 row(s) with missing quantity (12 row(s))
[qty negative] dropped 13 row(s) with negative quantity (refunds) (13 row(s))
revenue before drop: 4594.5
revenue after drop: 4740.0


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
print(df['item'].value_counts())

ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho',
}

before_items = df['item'].nunique()
df['item'] = df['item'].map(ITEM_MAP)
after_items = df['item'].nunique()

log('item', f'collapsed {before_items} spelling variants into {after_items} canonical products', len(df))
print(df['item'].value_counts())

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] collapsed 6 spelling variants into 3 canonical products (275 row(s))
item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
print(df['category'].value_counts())

df['category'] = df['category'].str.lower().str.strip().str.replace('-', '', regex=False)

CATEGORY_MAP = {
    'food': 'Food',
    'merch': 'Merch',
    'apparel': 'Merch',   # business decision: folding Apparel into Merch —
                            # not a spelling issue, a judgment call that
                            # a T-shirt/apparel item counts as merchandise
    'raingear': 'RainGear',
}
before_cats = df['category'].nunique()
df['category'] = df['category'].map(CATEGORY_MAP)
after_cats = df['category'].nunique()

log('category', f'normalized casing/punctuation, merged Apparel into Merch as a business decision — {before_cats} variants collapsed to {after_cats} categories', len(df))
print(df['category'].value_counts())

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[category] normalized casing/punctuation, merged Apparel into Merch as a business decision — 4 variants collapsed to 3 categories (275 row(s))
category
Food        95
Merch       94
RainGear    86
Name: count, dtype: int64


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert df['item'].isin(['Cheeseburger', 'Foam Finger', 'Rain Poncho']).all(), 'item should only contain the 3 canonical product names'
assert df['category'].isin(['Food', 'Merch', 'RainGear']).all(), 'category should only contain the 3 canonical categories'
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
df['revenue'] = df['qty'] * df['price']

by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2)
total_revenue = round(df['revenue'].sum(), 2)

print(by_category)
print()
print('total revenue:', total_revenue)

category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64

total revenue: 4740.0


**What I would tell the vendor: Whichever category comes out on top in by_category (likely Food, given it has the most spelling variants feeding into it and the highest base selection weight) is the one worth stocking more of, however this is only worth saying with the actual printed number in hand, since three roughly-equal-weighted categories in a 300-row seeded dataset can still land close together.

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,dropped exact duplicate rows,15
1,price,"stripped $ signs and whitespace, converted to ...",300
2,qty missing,dropped 12 row(s) with missing quantity,12
3,qty negative,dropped 13 row(s) with negative quantity (refu...,13
4,item,collapsed 6 spelling variants into 3 canonical...,275
5,category,"normalized casing/punctuation, merged Apparel ...",275


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a) The qty-cleaning step (TODO 3) changed revenue the most, though not in the direction you might expect. Before dropping missing/negative quantity rows, total revenue was \$4,594.50; after dropping 12 rows with missing quantity and 13 rows with negative quantity, it rose to \$4,740.00 — an increase of \$145.50. The missing-quantity rows didn't actually affect the total either way, since NaN × price also comes out as NaN and .sum() ignores those by default. The real driver was removing the negative-quantity rows: those represented refunds subtracting from revenue, so dropping them (rather than keeping them as reversed revenue) removed a negative contribution and pushed the total up. This was the single largest swing in the pipeline — the item and category cleanup steps only relabel text and never touch qty or price, so they can't move the revenue total at all.

b)The clearest judgment call I made was dropping the 13 negative-quantity rows (refunds) instead of keeping them as real, reversed revenue. A reasonable person could argue the opposite: a refund is a genuine business event, and treating it as if it never happened overstates how much money actually changed hands that day. Had I kept those rows instead of dropping them, reported revenue would have been \$4,594.50 rather than \$4,740.00 --> \$145.50 lower. I went with dropping because this lab's instructions explicitly directed dropping missing/negative qty rows rather than leaving that as an open decision, but in a real report I would flag this choice openly, since \$145.50 is not a small swing and a vendor relying on this number to plan inventory deserves to know which version they're looking at.